## Notebook Overview
## Deep Kernel Learning PART 1

- This notebook evaluates how effectively **Deep Kernel Learning (DKL)** models can predict **ISUP prostate cancer grade** using **radiomics features** extracted from prostate MRI. After loading and balancing the dataset (undersampling ISUP 0 and applying SMOTE), the notebook performs a **systematic hyperparameter search** over several DKL model components:

- Feature Set: We use **all available radiomics features**  
(107 features after preprocessing) since we checked the top 10 and top 20 correlated features but they did not improve the results.
- Neural Network Feature Extractors: These networks map radiomics features into a **lower‑dimensional latent space**.
- Activation Functions
- Latent Dimensions for the GP Kernel: These define the dimensionality of the learned feature space passed to the Gaussian Process.
- Gaussian Process Kernels  
- Noise Levels
- Learning Rates
    
For every combination of extractor, activation, latent dimension, kernel, noise value  
and learning rate we:

- Train a DKL model for **300 epochs**  
- Evaluate performance using **MSE** and **R²**  
- Compute **predictive uncertainty** for each test sample  
- Store all results  
- Identifie the **best‑performing configuration**  
- Reports **uncertainty per ISUP class**

Parts of the code were adapted and modified from the GPyTorch documentation: https://docs.gpytorch.ai/en/stable/examples/06_PyTorch_NN_Integration_DKL/KISSGP_Deep_Kernel_Regression_CUDA.html

In [1]:

import torch
import gpytorch
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from deep_gp.preprocessing_data import load_data, undersample_class0, apply_smote
from deep_gp.deep_kernel_class import GPRegressionModel
from tqdm import tqdm


import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))



%matplotlib inline
%load_ext autoreload
%autoreload 2



In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:
data = load_data()
df_new = undersample_class0(data)
df_resampled = apply_smote(df_new)

X_balanced = df_resampled.drop(columns=["case_ISUP"])
y_balanced = df_resampled["case_ISUP"]
all_features = df_resampled.drop(columns=["case_ISUP"]).columns.tolist()


In [4]:
def run_dkl_experiment(feature_list, df,
                       latent_dim=10,
                       extractor_type="large",
                       activation="relu",
                       kernel_type="rbf_ard",
                       noise_value=0.05,
                       learning_rate=0.01,
                       n_epochs=300):

    print(
    f"\nRunning DKL with {len(feature_list)} features, "
    f"latent_dim={latent_dim}, extractor={extractor_type}, "
    f"activation={activation}, kernel={kernel_type}, "
    f"noise={noise_value}, lr={learning_rate}"
)

    X = df[feature_list]
    y = df["case_ISUP"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled  = scaler_X.transform(X_test)

    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1,1)).ravel()
    y_test_scaled  = scaler_y.transform(y_test.values.reshape(-1,1)).ravel()
    
    train_x = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
    train_y = torch.tensor(y_train_scaled, dtype=torch.float32).to(device)
    test_x  = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
    test_y  = torch.tensor(y_test_scaled, dtype=torch.float32).to(device)

    
    likelihood = gpytorch.likelihoods.GaussianLikelihood()


    # Case 1: homoskedastic GP (learn noise automatically)
    if noise_value == "learned":
        model = GPRegressionModel(
            train_x, train_y, likelihood,
            data_dim=train_x.shape[1],
            latent_dim=latent_dim,
            extractor_type=extractor_type,
            activation=activation,
            kernel_type=kernel_type,
            noise_value=None   # tell the model not to override noise
        )

    # Case 2: fixed-noise GP (heteroskedastic search)
    else:
        model = GPRegressionModel(
            train_x, train_y, likelihood,
            data_dim=train_x.shape[1],
            latent_dim=latent_dim,
            extractor_type=extractor_type,
            activation=activation,
            kernel_type=kernel_type,
            noise_value=noise_value
        )

    model = model.to(device)
    likelihood = likelihood.to(device)



    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    model.train()
    likelihood.train()

    for i in range(n_epochs):
        optimizer.zero_grad()
        output = model(train_x)
        loss = -mll(output, train_y)
        loss.backward()
        optimizer.step()

    model.eval()
    likelihood.eval()

    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        preds = likelihood(model(test_x))

    # Predictions
    y_pred = scaler_y.inverse_transform(preds.mean.cpu().numpy().reshape(-1,1)).ravel()
    y_true = scaler_y.inverse_transform(test_y.cpu().numpy().reshape(-1,1)).ravel()

    # compute predictive uncertainty
    f_std = preds.stddev.cpu().numpy().ravel()

    # Build uncertainty dataframe
    df_unc = pd.DataFrame({
        "true": y_true,
        "pred": y_pred,
        "std": f_std
    })
    df_unc["true_class"] = np.round(df_unc["true"]).astype(int)

    # Metrics
    mse = mean_squared_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)

    print(f"MSE={mse:.4f}, R²={r2:.4f}")

    return mse, r2, df_unc


In [5]:
feature_sets = {
    "all": all_features
}

extractors = ["small", "medium", "large", "dkl"]
activations = ["relu", "tanh"]
latent_dims = [5, 15, 20]
kernels = ["matern_15", "rbf_ard"]
noise_values = ["learned", 0.01, 0.05, 0.1]
learning_rates = [0.01, 0.005]

results = []

# Count total runs for tqdm
total_runs = (
    len(feature_sets)
    * len(extractors)
    * len(activations)
    * len(latent_dims)
    * len(kernels)
    * len(noise_values)
    * len(learning_rates)
)

pbar = tqdm(total=total_runs, desc="DKL Hyperparameter Search", ncols=100)

for feat_name, feat_list in feature_sets.items():
    print(f"\n==============================")
    print(f"   Testing feature set: {feat_name}")
    print(f"==============================")

    for ext in extractors:
        for act in activations:
            for ld in latent_dims:
                for kernel in kernels:
                    for noise in noise_values:
                        for lr in learning_rates:

                            pbar.set_postfix({
                                "feat": feat_name,
                                "ext": ext,
                                "ld": ld,
                                "act": act,
                                "kernel": kernel,
                                "noise": noise,
                                "lr": lr
                            })

                            mse, r2, df_unc = run_dkl_experiment(
                                feature_list=feat_list,
                                df=df_resampled,
                                latent_dim=ld,
                                extractor_type=ext,
                                activation=act,
                                kernel_type=kernel,
                                noise_value=noise,
                                learning_rate=lr
                            )

                            results.append({
                                "features": feat_name,
                                "extractor": ext,
                                "activation": act,
                                "latent_dim": ld,
                                "kernel": kernel,
                                "noise": noise,
                                "lr": lr,
                                "mse": mse,
                                "r2": r2,
                                "uncertainty": df_unc
                            })

                            pbar.update(1)

pbar.close()


DKL Hyperparameter Search:   0%| | 0/384 [00:00<?, ?it/s, feat=all, ext=small, ld=5, act=relu, kerne


   Testing feature set: all

Running DKL with 107 features, latent_dim=5, extractor=small, activation=relu, kernel=matern_15, noise=learned, lr=0.01


DKL Hyperparameter Search:   0%| | 1/384 [00:04<30:54,  4.84s/it, feat=all, ext=small, ld=5, act=rel

MSE=1.6434, R²=0.4849

Running DKL with 107 features, latent_dim=5, extractor=small, activation=relu, kernel=matern_15, noise=learned, lr=0.005


DKL Hyperparameter Search:   1%| | 2/384 [00:07<24:01,  3.77s/it, feat=all, ext=small, ld=5, act=rel

...

MSE=2.2824, R²=0.2846

Running DKL with 107 features, latent_dim=20, extractor=dkl, activation=tanh, kernel=rbf_ard, noise=0.1, lr=0.01


DKL Hyperparameter Search: 100%|▉| 383/384 [20:09<00:02,  2.95s/it, feat=all, ext=dkl, ld=20, act=ta

MSE=2.9379, R²=0.0791

Running DKL with 107 features, latent_dim=20, extractor=dkl, activation=tanh, kernel=rbf_ard, noise=0.1, lr=0.005


DKL Hyperparameter Search: 100%|█| 384/384 [20:12<00:00,  3.16s/it, feat=all, ext=dkl, ld=20, act=ta

MSE=1.8619, R²=0.4164


In [ ]:
results_sorted = sorted(results, key=lambda x: x["r2"], reverse=True)
best = results_sorted[0]

print("\n===== BEST MODEL FOUND =====")
print(f"Feature set:   {best['features']}")
print(f"Extractor:     {best['extractor']}")
print(f"Activation:    {best['activation']}")
print(f"Latent dim:    {best['latent_dim']}")
print(f"Noise:         {best['noise']}")
print(f"Kernel:        {best['kernel']}")
print(f"LR:            {best['lr']}")
print(f"MSE:           {best['mse']:.4f}")
print(f"R²:            {best['r2']:.4f}")

df_unc_best = best["uncertainty"]

unc_by_class = df_unc_best.groupby("true_class")["std"].mean()

print("\n===== MEAN UNCERTAINTY (STD) PER ISUP CLASS =====")
for cls, std in unc_by_class.items():
    print(f"ISUP {cls}:  std = {std:.4f}")



===== BEST MODEL FOUND =====
Feature set:   all
Extractor:     dkl
Activation:    relu
Latent dim:    15
Noise:         0.1
Kernel:        rbf_ard
LR:            0.005
MSE:           1.2402
R²:            0.6112

===== MEAN UNCERTAINTY (STD) PER ISUP CLASS =====
ISUP 0:  std = 0.3111
ISUP 1:  std = 0.3195
ISUP 2:  std = 0.3132
ISUP 3:  std = 0.2997
ISUP 4:  std = 0.2804
ISUP 5:  std = 0.2852
